# FINRL Walk-Forward Experiment

This notebook runs the direct-feature walk-forward experiment runner and visualizes portfolio performance against the S&P 500 / SPY benchmark.

Use small synthetic data locally. Use Colab for the full configured universe, direct portfolio optimization, and full walk-forward experiments.

In [ ]:
%cd /content
![ -d FINRL ] || git clone https://github.com/nidarshans/FINRL.git
%cd /content/FINRL

In [ ]:
%cd /content/FINRL
%pip install -e .


In [1]:
# Keep notebook imports pointed at the live workspace package.

from datetime import date, timedelta
from functools import lru_cache

import polars as pl

from finrl.backtest.walk_forward import WalkForwardConfig
from finrl.dpo_jax import DPOConfig
from finrl.data import (
    MarketDataBundle,
    MarketDataConfig,
    UniverseConfig,
    build_rebalance_calendar,
    compute_open_to_open_returns,
    download_ohlcv,
)
from finrl.data.download import download_macro_series
from finrl.env.trading_env import EnvConfig
from finrl.experiments import (
    ExperimentConfig,
    RawExperimentData,
    build_allocation_figure,
    build_drawdown_figure,
    build_holdings_heatmap_granular,
    build_performance_figure,
    build_regime_portfolio_figure,
    build_spectral_figure,
    metrics_to_frame,
    run_walk_forward_experiment,
    build_run_metadata,
    save_walk_forward_artifacts,
)
from finrl.features import FeatureConfig, FeatureTransformSpec, build_feature_bundle, selected_direct_allocation_indices
from finrl.features.preprocessing import PreprocessingConfig
from finrl.features.schema import FeatureBundle
import numpy as np

## Prepared Data Contract

The runner expects prepared feature and return tables:

- `FeatureBundle` with asset and macro features plus a dummy 20-column spectral compatibility table.
- `returns`: Polars DataFrame with `decision_date` and one return column per tradable asset.
- `spy_returns`: Polars DataFrame with `decision_date` and `spy_return` for the same holding periods.

Replace the synthetic fixture below with the output of the data, feature, preprocessing, and return-preparation pipeline for full experiments. DPO trains the direct allocation head over explicitly routed asset features.

## Run With Real yfinance Data

Edit `TICKERS`, `START`, `END`, and `MAX_STOCKS`, then run this section in Colab. The code downloads real stock or bond ticker data plus SPY, computes daily open-to-open returns, builds causal per-asset features, and packages everything into `RawExperimentData` for the walk-forward runner.

In [2]:
TICKERS = [
    "IEF", "GLD", "BIL", "LLY", "JNJ", "HWM", "HOOD", "COIN",
    "GOOG", "NET", "ZS", "CRWD", "PANW", "B", "QCOM", "NVDA", "MU",
    "AMD", "NFLX", "GME", "F", "KO", "MCD", "LMT", "COIN"
]

MAX_STOCKS = len(TICKERS)  # set to 100 after pasting your full universe
START = "2000-01-01"
END = "2026-07-25"
CACHE_DIR = "data/cache"
BENCHMARK_TICKER = "SPY"
REBALANCE_FREQUENCY = "daily"  # "daily" or "weekly"

universe = UniverseConfig(
    tickers=TICKERS,
    max_stocks=MAX_STOCKS,
    include_cash=False,
    benchmark_ticker=BENCHMARK_TICKER,
)
market_config = MarketDataConfig(
    universe=universe,
    start=START,
    end=END,
    cache_dir=CACHE_DIR,
)
selected_tickers = universe.selected_tickers
selected_tickers

('IEF',
 'GLD',
 'BIL',
 'LLY',
 'JNJ',
 'HWM',
 'HOOD',
 'COIN',
 'GOOG',
 'NET',
 'ZS',
 'CRWD',
 'PANW',
 'B',
 'QCOM',
 'NVDA',
 'MU',
 'AMD',
 'NFLX',
 'GME',
 'F',
 'KO',
 'MCD',
 'LMT')

In [3]:
def _returns_wide(open_to_open_returns: pl.DataFrame, tickers: tuple[str, ...]) -> pl.DataFrame:
    wide = (
        open_to_open_returns
        .select(["decision_date", "ticker", "return"])
        .pivot(index="decision_date", on="ticker", values="return", aggregate_function="first")
        .sort("decision_date")
    )
    return wide.select(["decision_date", *tickers]).fill_null(0.0)


def _spy_returns(open_to_open_returns: pl.DataFrame) -> pl.DataFrame:
    return (
        open_to_open_returns
        .select(["decision_date", pl.col("return").alias("spy_return")])
        .sort("decision_date")
        .drop_nulls()
    )


def _filter_features_to_common_dates(features: FeatureBundle, returns: pl.DataFrame, spy_returns: pl.DataFrame) -> FeatureBundle:
    common_dates = (
        returns.select("decision_date")
        .join(spy_returns.select("decision_date"), on="decision_date", how="inner")
        .rename({"decision_date": "date"})
        .with_columns(pl.col("date").cast(pl.Date))
        .unique()
        .sort("date")
    )
    asset = features.asset_features.join(common_dates, on="date", how="inner").sort(["date", "ticker"])
    macro = (
        common_dates
        .join(features.macro_features, on="date", how="left")
        .sort("date")
        .with_columns(pl.all().exclude("date").forward_fill().fill_null(0.0))
    )
    spectral = features.spectral_features.join(common_dates, on="date", how="inner").sort("date")
    dates = tuple(common_dates.get_column("date").to_list())
    return FeatureBundle(
        asset_features=asset,
        macro_features=macro,
        spectral_features=spectral,
        decision_dates=dates,
        tickers=features.tickers,
        asset_feature_columns=features.asset_feature_columns,
        macro_feature_columns=features.macro_feature_columns,
        spectral_feature_columns=features.spectral_feature_columns,
    )


@lru_cache(maxsize=1)
def load_real_market_inputs() -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    ohlcv = download_ohlcv(selected_tickers, START, END, market_config)
    spy_ohlcv = download_ohlcv((BENCHMARK_TICKER,), START, END, market_config)
    macro = download_macro_series(START, END, market_config)
    calendar = build_rebalance_calendar(ohlcv, REBALANCE_FREQUENCY)
    return ohlcv, spy_ohlcv, macro, calendar


def make_real_yfinance_data(feature_set: str = "baseline_current_14") -> RawExperimentData:
    ohlcv, spy_ohlcv, macro, calendar = load_real_market_inputs()

    market_bundle = MarketDataBundle(
        ohlcv=ohlcv,
        spy_ohlcv=spy_ohlcv,
        macro=macro,
        calendar=calendar,
    )
    features = build_feature_bundle(
        market_bundle,
        FeatureConfig(
            accumulation_window=40,
            klinger_fast_span=34,
            klinger_slow_span=55,
            klinger_signal_span=13,
            macd_fast_span=12,
            macd_slow_span=26,
            macd_signal_span=9,
            mr_ewma_span=200,
            mr_vol_window=200,
            spectral_dim=20,
            cmf_window=60,
            feature_set=feature_set,
        ),
    )

    stock_returns = _returns_wide(
        compute_open_to_open_returns(ohlcv, calendar),
        selected_tickers,
    )
    spy_returns = _spy_returns(compute_open_to_open_returns(spy_ohlcv, calendar))
    features = _filter_features_to_common_dates(features, stock_returns, spy_returns)
    common_dates = pl.DataFrame({"decision_date": list(features.decision_dates)}).with_columns(pl.col("decision_date").cast(pl.Date))
    stock_returns = common_dates.join(stock_returns, on="decision_date", how="inner")
    spy_returns = common_dates.join(spy_returns, on="decision_date", how="inner")
    return RawExperimentData(features=features, returns=stock_returns, spy_returns=spy_returns)


raw_data = make_real_yfinance_data()
raw_data.features.asset_features.tail(), raw_data.returns.tail(), raw_data.spy_returns.tail()

(shape: (5, 16)
 ┌────────────┬────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
 │ date       ┆ ticker ┆ mr_ewma50_ ┆ ewma50_sl ┆ … ┆ cmf_days_ ┆ frog_in_t ┆ bollinger ┆ fip_over_ │
 │ ---        ┆ ---    ┆ vol_gap    ┆ ope       ┆   ┆ since_cro ┆ he_pan    ┆ _bandwidt ┆ bollinger │
 │ date       ┆ str    ┆ ---        ┆ ---       ┆   ┆ ss        ┆ ---       ┆ h         ┆ _bandwidt │
 │            ┆        ┆ f64        ┆ f64       ┆   ┆ ---       ┆ f64       ┆ ---       ┆ h         │
 │            ┆        ┆            ┆           ┆   ┆ i64       ┆           ┆ f64       ┆ ---       │
 │            ┆        ┆            ┆           ┆   ┆           ┆           ┆           ┆ f64       │
 ╞════════════╪════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
 │ 2026-07-22 ┆ NFLX   ┆ 13.64735   ┆ -4.127923 ┆ … ┆ 37        ┆ -1.897367 ┆ 0.155217  ┆ -12.22397 │
 │ 2026-07-22 ┆ NVDA   ┆ -4.21908   ┆ 2.058459  ┆ … ┆ 27        ┆ 

In [4]:
# Each feature set is built independently from the same raw data and then
# evaluated with identical walk-forward settings.  Keep this list locked
# before looking at out-of-sample results.
FEATURE_SETS = (
    "baseline_plus_volume_ema_volatility",
)

'''
"baseline_current_14",
"baseline_plus_momentum",
"baseline_plus_momentum_ranked",
"baseline_plus_risk",
"baseline_plus_liquidity",
"baseline_plus_liquidity_ranked",
"baseline_plus_structure",
"baseline_plus_market_relative",
"institutional_core_v1",
"institutional_core_plus_structure_v1",
"baseline_plus_volume_ema_volatility" 
"baseline_plus_volume_ema"
'''

# DPO uses one complete chronological scan per epoch; there is no batch-size setting.
# After the initial fit, DPO retrains on an expanding cumulative window each year.
# Later fits use shrink-and-perturb initialization and reset Adam state.
# Training uses dense allocations; evaluation keeps only the top-N positions.
# This top-N projection is intentionally evaluation-only.
# DPO minimizes negative net-return Sharpe plus mean drawdown excess.
# Transaction costs remain disabled for this zero-cost benchmark.
POLICY_MODE = "dpo"  # "dpo" or "equal_weight"
TOP_N_POSITIONS = None
DRAWDOWN_LIMIT = 0.05
DRAWDOWN_PENALTY = 1
SHRINK_PERTURB_SHRINK_FACTOR = 0.1
SHRINK_PERTURB_PERTURB_SCALE = 0.5

# Explicit feature transforms keep bounded/event features interpretable and
# use prior observations only for the selected surprise-style signals.
FEATURE_TRANSFORMS = (
    FeatureTransformSpec("mr_ewma50_vol_gap", "clipped_passthrough", clip_lower=-10.0, clip_upper=10.0),
    FeatureTransformSpec("ewma50_slope", "clipped_passthrough", clip_lower=-10.0, clip_upper=10.0),
    FeatureTransformSpec("cmf", "clipped_passthrough", clip_lower=-1.0, clip_upper=1.0),
    FeatureTransformSpec("cmf_cross_signal", "passthrough"),
    FeatureTransformSpec("cmf_days_since_cross", "clipped_passthrough", clip_lower=0.0, clip_upper=252.0),
    FeatureTransformSpec("volume_z_20", "lagged_rolling_zscore", rolling_periods=252),
)

def make_experiment_config(feature_set: str, transaction_cost_bps: float) -> ExperimentConfig:
    return ExperimentConfig(
        walk_forward=WalkForwardConfig(
            train_years=3,
            test_years=1,
            step_years=1,
            expanding_train_window=POLICY_MODE == "dpo",
        ),
        preprocessing=PreprocessingConfig(rolling_window=252, feature_transforms=FEATURE_TRANSFORMS),
        dpo=DPOConfig(
            learning_rate=1e-4,
            num_epochs=20,
            shrink_perturb_shrink_factor=SHRINK_PERTURB_SHRINK_FACTOR,
            shrink_perturb_perturb_scale=SHRINK_PERTURB_PERTURB_SCALE,
            transaction_cost_bps=transaction_cost_bps,
            drawdown_limit=DRAWDOWN_LIMIT,
            drawdown_penalty=DRAWDOWN_PENALTY,
            allocation_hidden_dims=(32, 16, 8),
            allocation_hidden_activation="tanh",
            allocation_output_activation="identity",
            allocation_use_layer_norm=True,
            simplex_activation="sparsemax",
        ),
        env=EnvConfig(
            drawdown_limit=DRAWDOWN_LIMIT,
            drawdown_penalty=DRAWDOWN_PENALTY,
            sortino_target_return=0.0,
            sortino_downside_penalty=0.0,
            top_n_positions=TOP_N_POSITIONS,
            transaction_cost_rate=transaction_cost_bps / 10_000.0,
        ),
        enable_dpo=POLICY_MODE == "dpo",
        rebalance_frequency=REBALANCE_FREQUENCY,
        seed=9,
        feature_set=feature_set,
    )

COST_BPS = 0.0  # Zero-cost benchmark; DPO and environment use the same rate.
ablation_results = {}
ablation_data = {}
ablation_metrics = []
for feature_set in FEATURE_SETS:
    experiment_data = make_real_yfinance_data(feature_set)
    config = make_experiment_config(feature_set, COST_BPS)
    routing = selected_direct_allocation_indices(
        experiment_data.features.asset_feature_columns, feature_set
    )
    result = run_walk_forward_experiment(experiment_data, config)
    ablation_results[feature_set] = result
    ablation_data[feature_set] = experiment_data
    ablation_metrics.append(
        metrics_to_frame(result).with_columns(pl.lit(feature_set).alias("feature_set"))
    )
    print({
        "feature_set": feature_set,
        "stocks": len(experiment_data.features.tickers),
        "direct_features": len(routing.direct_allocation_indices),
        "decision_dates": len(experiment_data.features.decision_dates),
    })

ablation_summary = pl.concat(ablation_metrics, how="vertical")
display(ablation_summary)

{'feature_set': 'baseline_plus_volume_ema_volatility', 'stocks': 24, 'direct_features': 14, 'decision_dates': 6677}


split_index,test_start,test_end,portfolio_cumulative_return,spy_cumulative_return,spy_relative_alpha,portfolio_max_drawdown,portfolio_mean_turnover,portfolio_total_transaction_cost,portfolio_sharpe_ratio,portfolio_sortino_ratio,portfolio_calmar_ratio,tracking_error,information_ratio,beta,regression_alpha,feature_set
i64,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
0,2003-01-01,2003-12-31,-0.036101,0.248611,-0.284712,0.059134,0.081952,0.0,-0.722003,-0.975014,-0.610494,0.153963,-1.772067,0.149596,-0.071049,"""baseline_plus_volume_ema_volat…"
1,2004-01-01,2004-12-31,0.134886,0.099175,0.035711,0.232837,0.219828,0.0,0.633532,0.840592,0.579314,0.195693,0.296318,1.563145,0.001529,"""baseline_plus_volume_ema_volat…"
2,2005-01-01,2005-12-31,0.0,0.071703,-0.071703,0.0,0.007937,0.0,0.0,0.0,0.0,0.095703,-0.771541,0.0,0.0,"""baseline_plus_volume_ema_volat…"
3,2006-01-01,2006-12-31,-0.005543,0.133861,-0.139404,0.010678,0.040025,0.0,-0.550879,-0.630037,-0.521146,0.09381,-1.452379,0.02253,-0.008475,"""baseline_plus_volume_ema_volat…"
4,2007-01-01,2007-12-31,0.183694,0.044991,0.138703,0.082368,0.361862,0.0,1.118399,1.627621,2.239832,0.113561,1.128731,0.832901,0.137292,"""baseline_plus_volume_ema_volat…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
19,2022-01-01,2022-12-31,-0.257775,-0.187437,-0.070338,0.283238,0.448024,0.0,-0.553658,-0.779089,-0.913212,0.255753,-0.159732,1.300535,0.013041,"""baseline_plus_volume_ema_volat…"
20,2023-01-01,2023-12-31,0.411173,0.246359,0.164813,0.133832,0.47499,0.0,1.563824,2.393355,3.101401,0.160752,0.903333,1.38577,0.056049,"""baseline_plus_volume_ema_volat…"
21,2024-01-01,2024-12-31,0.399527,0.264942,0.134586,0.232194,0.43595,0.0,1.309337,1.984348,1.720659,0.227549,0.592417,1.495655,0.014317,"""baseline_plus_volume_ema_volat…"


In [5]:
# Select one result for the charts below only after reviewing the full table.
SELECTED_FEATURE_SET = "baseline_plus_volume_ema_volatility"
result = ablation_results[SELECTED_FEATURE_SET]
selected_config = make_experiment_config(SELECTED_FEATURE_SET, COST_BPS)
artifact_dir = f"walk_forward_artifacts/{SELECTED_FEATURE_SET}_zero_cost"
save_walk_forward_artifacts(
    result,
    build_run_metadata(ablation_data[SELECTED_FEATURE_SET], selected_config),
    artifact_dir,
)
print(f"Saved selected-run artifacts to {artifact_dir}")

Saved selected-run artifacts to walk_forward_artifacts/baseline_plus_volume_ema_volatility_zero_cost


## Performance vs S&P 500


In [6]:
performance_fig = build_performance_figure(result)
performance_fig.show()


## Drawdown

Compare portfolio and SPY peak-to-trough declines over the walk-forward period.


In [7]:
drawdown_fig = build_drawdown_figure(result)
drawdown_fig.show()


## Portfolio Allocation


In [8]:
allocation_fig = build_allocation_figure(result)
allocation_fig.show()


## Holdings Heatmap


In [9]:
holdings_heatmap_fig = build_holdings_heatmap_granular(
    result,
    min_weight=0.001,
    top_n=50,
    freq=None,
    include_cash=False,
)
holdings_heatmap_fig.show()
